## 附录代码（中等精简，约 1 页 Word）

较「一页极简」仅多：**基线 LSTM** 定义，便于与主模型对照。

仍不展开：按日合并 LLM、`G1`～`G3-M` 列名与标准化、FinBERT 推理循环、训练早停与随机搜索的完整实现——见 `99_appendix_consolidated_code.ipynb` 与 `notebooks/1–4`。


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset


def build_target_sigma_t_plus_1(df: pd.DataFrame) -> pd.DataFrame:
    """次日对数 Parkinson 波动率；缺失仅用前向填充（防穿越）。"""
    d = df.ffill()
    H, L = d["high"], d["low"]
    Vp = (np.log(H / L)) ** 2 / (4 * np.log(2))
    sigma = np.log(Vp + 1e-8)
    d = d.copy()
    d["target_sigma_t_plus_1"] = pd.Series(sigma).shift(-1)
    return d.dropna(subset=["target_sigma_t_plus_1"]).reset_index(drop=True)


class CryptoVolatilityDataset(Dataset):
    """lookback=7：用 t-6..t 预测 sigma_{t+1}（与主实验对齐）。"""

    def __init__(self, X: np.ndarray, y: np.ndarray, window: int = 7):
        self.X, self.y, self.w = X, y, window

    def __len__(self) -> int:
        return len(self.X) - self.w + 1

    def __getitem__(self, i: int):
        x = torch.tensor(self.X[i : i + self.w], dtype=torch.float32)
        y = torch.tensor([self.y[i + self.w - 1]], dtype=torch.float32)
        return x, y


In [ ]:
class BaselineLSTM(nn.Module):
    """对照：单向 LSTM，末步隐状态经 Dropout 后接两层 MLP。"""

    def __init__(self, n_feat: int, hidden: int, dropout: float):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, num_layers=1, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        return self.head(self.drop(out[:, -1, :]))


class BiLSTMAttention(nn.Module):
    """主模型：双向 LSTM + 注意力池化 + MLP 头。"""

    def __init__(self, n_feat: int, hidden: int, dropout: float):
        super().__init__()
        self.rnn = nn.LSTM(n_feat, hidden, batch_first=True, bidirectional=True)
        self.att = nn.Linear(hidden * 2, 1, bias=False)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden * 2, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x: torch.Tensor):
        h, _ = self.rnn(x)
        a = F.softmax(self.att(h), dim=1)
        c = (a * h).sum(dim=1)
        return self.head(self.drop(c)), a.squeeze(-1)


**训练与特征（不展开代码）**：在已标准化的特征矩阵上，按 `7:1:2` 划分训练/验证/测试；批量 MSE，`AdamW` + `ReduceLROnPlateau`，验证集早停并回载最优权重；特征组 `G1`～`G3-M` 在基础列上按正文表追加 FinBERT / LLM 因子。测试集 RMSE、MAE、R² 以仓库 `results` 中归档 CSV 为准。
